In [ ]:
import requests
import json
from datetime import datetime, timezone
from pyspark.sql import Row

In [ ]:
env='prod'
print(f"Running on environment: {env}")

In [ ]:
def configurations():
    day = datetime.now(timezone.utc).strftime("%m/%d/%Y")
    url_V1 = 'https://statsapi.mlb.com/api/v1/'
    url = f'{url_V1}schedule?sportId=1&date={day}'
    bronze_schema = f'mlb_{env}_bronze'
    table_schedule = f"{bronze_schema}.game_schedule"
    table_failed_game_schedule = f"{bronze_schema}.failed_game_schedule"
    return url, table_schedule, table_failed_game_schedule

url, table_schedule, table_failed_game_schedule = configurations()
url, table_schedule, table_failed_game_schedule

In [ ]:
def get_schedule(url):
    schedule = requests.get(url)
    try:
        schedule.raise_for_status()
        schedule = schedule.json().get('dates', [{}])[0].get('games', [])
        return schedule
    except requests.exceptions.HTTPError as err:
        raise err

schedule = get_schedule(url)
schedule

In [ ]:
def process_schedule(schedule):
    games_today = []
    failed_responses = []
    for game in schedule:
        try:
            game_id = str(game['gamePk'])
            home_team = game['teams']['home']['team']['name']
            away_team = game['teams']['away']['team']['name']
            game_scheduled_time = game['gameDate']
            game_scheduled_time = datetime.strptime(game_scheduled_time, "%Y-%m-%dT%H:%M:%SZ")
            status = game['status']['abstractGameCode']
            ingestion_time = datetime.now(timezone.utc)
            games_today.append(
                Row(
                    game_pk=game_id,
                    home_team=home_team,
                    away_team=away_team,
                    game_scheduled_time=game_scheduled_time,
                    status=status,
                    ingestion_timestamp=ingestion_time
                )
            )
        except Exception as e:
            failed_responses.append(
                Row(
                    response=json.dumps(game),
                    ingestion_timestamp=datetime.now(timezone.utc)
                )
            )
    return games_today, failed_responses

games_today, failed_responses = process_schedule(schedule)

In [0]:
def save_to_bronze(games_today, failed_responses, table_schedule, table_failed_game_schedule):
    if games_today:
        df_games = spark.createDataFrame(games_today)
        df_games.write.format("delta").mode("append").saveAsTable(table_schedule)
    if failed_responses:
        df_failed = spark.createDataFrame(failed_responses)
        df_failed.write.format("delta").mode("append").saveAsTable(table_failed_game_schedule)